# Import Packages

In [67]:
import asyncio
import logging
from collections.abc import AsyncIterable
from typing import Any, Dict

import artkit.api as ak
import openai

# from pilots.spc.constants.prompts.base_prompts import SPCDefaultPrompts
import nest_asyncio
import os
import pandas as pd

# Set up for environment (API key)

In [68]:
OPENAI_API_KEY= {Your_API_Key}
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
nest_asyncio.apply()

# Simple Test for API Call

In [36]:
import artkit.api as ak

# Set up a chat system with the OpenAI GPT-4o model
chat_llm = ak.CachedChatModel(
    model=ak.OpenAIChat(model_id="gpt-4o"),
    database="cache/chat_llm.db"
)

In [37]:

# A function that rephrases input prompts to have a specified tone
async def rephrase_tone(prompt: str, tone: str, llm: ak.ChatModel):

    response = await llm.get_response(
        message = (
            f"Your job is to rephrase in input question to have a {tone} tone.\n"
            f"This is the question you must rephrase:\n{prompt}"
        )
    )

    yield {"prompt": response[0], "tone": tone}


# A function that behaves as a chatbot named AskChad who mirrors the user's tone
async def ask_chad(prompt: str, llm: ak.ChatModel):

    response = await llm.get_response(
        message = (
            "You are AskChad, a chatbot that mirrors the user's tone. "
            "For example, if the user is rude, you are rude. "
            "Your responses contain no more than 10 words.\n"
            f"Respond to this user input:\n{prompt}"
        )
    )

    yield {"response": response[0]}


# A function that evaluates responses according to a specified metric
async def evaluate_metric(response: str, metric: str, llm: ak.ChatModel):

    score = await llm.get_response(
        message = (
            f"Your job is to evaluate prompts according to whether they are {metric}. "
            f"If the input prompt is {metric}, return 1, otherwise return 0.\n"
            f"Please evaluate the following prompt:\n{response}"
        )
    )

    yield {"evaluation_metric": metric, "score": int(score[0])}

# Simple Artkit pipeline test

In [50]:
pipeline = (
    ak.chain(
        ak.parallel(
            ak.step("tone_rephraser", rephrase_tone, tone="POLITE", llm=chat_llm),
            ak.step("tone_rephraser", rephrase_tone, tone="SARCASTIC", llm=chat_llm),
        ),
        ak.step("ask_chad", ask_chad, llm=chat_llm),
        ak.step("evaluation", evaluate_metric, metric="SARCASTIC", llm=chat_llm)
    )
)

prompt = {"prompt": "What is a fun activity to do in Boston?"}

# Run pipeline
result = ak.run(steps=pipeline, input=prompt)

# Convert results dictionary into a multi-column dataframe
result.to_frame()


INFO:fluxus.functional.conduit._consumer:Run took 0.0149367 seconds, with 0.0111627 seconds of total wait time. Concurrent execution achieved a speedup factor of 0.747329 over sequential execution.


input  \
                                       prompt   
item                                            
0     What is a fun activity to do in Boston?   
0     What is a fun activity to do in Boston?   

                                         tone_rephraser             \
                                                 prompt       tone   
item                                                                 
0     Could you please suggest a fun activity to enj...     POLITE   
0     Oh, what thrilling activity could one possibly...  SARCASTIC   

                                               ask_chad        evaluation  \
                                               response evaluation_metric   
item                                                                        
0     Sure! Try kayaking on the Charles River. Fun a...         SARCASTIC   
0     Try museums, historic tours, or vibrant neighb...         SARCASTIC   

            
     score  
item        
0        0  
0        0

# Long prompt but within token max --> Works fine

In [51]:
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

async def simulate_timeout(prompt: str, llm: ak.ChatModel):
    try:
        # Explicitly set a very short timeout to force timeout errors
        response = await llm.get_response(
            message=prompt,
            timeout=0.01  # Intentionally low timeout
        )
        return {"response": response[0]}
    except Exception as e:
        logger.error(f"Timeout error simulated: {e}")
        return {"response": "Timeout error occurred"}


test_prompt_1 = {
    "prompt": (
        "Imagine you are designing an exhaustive guide for visitors to a city with diverse attractions. "
        "The guide must cover activities for various interests: "
        "history enthusiasts, food lovers, outdoor adventurers, art connoisseurs, and families with children. "
        "For each group, describe an ideal itinerary that spans an entire day, including morning, afternoon, and evening activities. "
        "Each activity should include engaging descriptions, highlight why it is unique, and provide logistical details like timing and location. "
        "Additionally, rephrase each activity description twice: first in a formal, professional tone, "
        "and second in a humorous and sarcastic tone. Ensure that each description is no longer than 50 words. "
        "Finally, evaluate the sarcastic tone for consistency and provide a sarcasm adherence score (0 or 1). "
        * 800  # Repeat this pattern 
    )
}

# Run pipeline
result = ak.run(steps=pipeline, input=test_prompt_1)

# Convert results dictionary into a multi-column dataframe
result.to_frame()

INFO:fluxus.functional.conduit._consumer:Run took 0.0152513 seconds, with 0.011788 seconds of total wait time. Concurrent execution achieved a speedup factor of 0.772918 over sequential execution.


input  \
                                                 prompt   
item                                                      
0     Imagine you are designing an exhaustive guide ...   
0     Imagine you are designing an exhaustive guide ...   

                                         tone_rephraser             \
                                                 prompt       tone   
item                                                                 
0     I understand you're planning to design a detai...     POLITE   
0     Oh, because planning a day jam-packed with meg...  SARCASTIC   

                                               ask_chad        evaluation  \
                                               response evaluation_metric   
item                                                                        
0     Sounds interesting, but that's way too detaile...         SARCASTIC   
0     Exactly, who needs simplicity when complexity'...         SARCASTIC   

            
     score  
item        
0        1  
0        1

# Long prompt larger than token max --> error code 400

In [52]:
# Input to run through the pipeline

test_prompt_2 = {
    "prompt": (
        "Imagine you are designing an exhaustive guide for visitors to a city with diverse attractions. "
        "The guide must cover activities for various interests: "
        "history enthusiasts, food lovers, outdoor adventurers, art connoisseurs, and families with children. "
        "For each group, describe an ideal itinerary that spans an entire day, including morning, afternoon, and evening activities. "
        "Each activity should include engaging descriptions, highlight why it is unique, and provide logistical details like timing and location. "
        "Additionally, rephrase each activity description twice: first in a formal, professional tone, "
        "and second in a humorous and sarcastic tone. Ensure that each description is no longer than 50 words. "
        "Finally, evaluate the sarcastic tone for consistency and provide a sarcasm adherence score (0 or 1). "
        * 1000  # Repeat this pattern 100 times
    )
}

result = ak.run(steps=pipeline, input=test_prompt_2)

result.to_frame()

INFO:openai._base_client:Retrying request to /chat/completions in 0.464903 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 400 Bad Request"


BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 128000 tokens. However, your messages resulted in 151034 tokens. Please reduce the length of the messages.", 'type': 'invalid_request_error', 'param': 'messages', 'code': 'context_length_exceeded'}}

# Timeout < response time --> Get runtime error

In [ ]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


async def simulate_timeout(prompt: str, llm: ak.ChatModel):
    try:
        # Intentionally low timeout to replicate timeout error
        response = await llm.get_response(
            message=prompt,
            timeout=1  # Set a very short timeout
        )
        return {"response": response[0]}
    except Exception as e:
        logger.error(f"Timeout error simulated: {e}")
        return {"response": "Timeout error occurred"}

pipeline_2 = ak.chain(
    ak.step(
        "simulate_timeout",
        simulate_timeout,
        llm=chat_llm)
    )

# Run pipeline
result = ak.run(steps=pipeline_2, input=test_prompt_1)

# Convert results dictionary into a multi-column dataframe
result.to_frame()



INFO:openai._base_client:Retrying request to /chat/completions in 0.474264 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 0.905922 seconds
ERROR:__main__:Timeout error simulated: Request timed out.
INFO:fluxus.functional.conduit._consumer:Run took 5.22206 seconds, with 5.22087 seconds of total wait time. Concurrent execution achieved a speedup factor of 0.999771 over sequential execution.


,input,simulate_timeout
,prompt,response
item,,
0,Imagine you are designing an exhaustive guide ...,Timeout error occurred


# Streaming Function Test

# AsyncOpenAI

In [93]:
import openai
import asyncio

# Set up OpenAI API key
openai.api_key = OPENAI_API_KEY

# Long prompt for testing
long_prompt = """
You are an AI assistant tasked with generating an extensive and multifaceted document on the following topics:
1. **Interplanetary Political Systems**: A 3000-word analysis of the political structures governing a federation of 50 planets, including:
   - Detailed descriptions of each planet's unique political system.
   - The process of electing interplanetary representatives and resolving conflicts.
   - Case studies of three hypothetical political crises and their resolutions.
2. **Interstellar Economy**: A 3000-word detailed breakdown of:
   - The trade networks, currency systems, and taxation policies across planets.
   - The effects of hyperinflation and interstellar market crashes, with proposed recovery plans.
   - A mathematical model (with equations) illustrating the balance of trade and resource allocation.
3. **Cultural Exchange**: A 3000-word discussion of:
   - Linguistic, artistic, and religious exchanges among the planets.
   - The role of interplanetary festivals and their impact on diplomacy.
   - Detailed character-driven stories illustrating cultural misunderstandings and resolutions.
4. **Technological Innovations**: A 3000-word technical overview of:
   - Space travel advancements, including propulsion systems, quantum communication, and AI-assisted navigation.
   - The evolution of terraforming technologies, with case studies of three planets.
   - A technical schematic of a futuristic space station, complete with annotations for key modules.
5. **Creative Writing Task**: Write a 5000-word science fiction story:
   - The story should feature five well-developed characters from different planets.
   - Include a high-stakes interstellar diplomatic mission that goes awry.
   - Use rich, vivid descriptions of planets, space stations, and alien ecosystems.
   - Integrate political intrigue, action sequences, and philosophical debates about humanity's future.

Ensure each section is written with extreme detail, integrating plausible science, hypothetical scenarios, and deep narrative elements. Connect the sections cohesively so that they reflect a single, unified universe.
"""


async_client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)

async def get_response_without_streaming():
    try:
        # Directly request a very large and complex response
        response = await async_client.chat.completions.create(
            model="gpt-4",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": long_prompt}
            ],
            timeout = 10
        )
        print("Response received (non-streaming):")
        print(response.choices[0].message.content)
    except Exception as e:
        print(f"Error without streaming: {e}")


# Function for streaming response
async def get_response_with_streaming():
    try:
        # Use stream=True for streaming response
        response = await async_client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": long_prompt}
            ],
            stream=True
        )
        print("Streaming response:")
        async for chunk in response:
            # Extract the content from the streamed chunk
            content = chunk.choices[0].delta.content
            print(content, end='', flush=True)
    except Exception as e:
        print(f"Error with streaming: {e}")

# Main function to test both methods
async def main():
    print("\nWithout streaming:")
    await get_response_without_streaming()

    print("\n\nWith streaming:")
    await get_response_with_streaming()

# Run the main function
asyncio.run(main())





Without streaming:


INFO:openai._base_client:Retrying request to /chat/completions in 0.446455 seconds
INFO:openai._base_client:Retrying request to /chat/completions in 0.907431 seconds


Error without streaming: Request timed out.


With streaming:


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Streaming response:
### Interplanetary Political Systems

#### Introduction
In a vast federation of 50 planets, diverse political systems coalesce under a loose constitutional framework that seeks to harmonize interplanetary governance while preserving the unique identities of each world. Each planet is endowed with autonomy to maintain its internal political structure, representing everything from democracies and oligarchies to technocracies and monarchies. This document explores the complexities of these systems, the election of interplanetary representatives, conflict resolution mechanisms, and provides case studies of crises and their resolutions.

#### Detailed Descriptions of Planetary Political Systems
1. **Democratic Republic of Veridia**:
   - Veridia follows a representative democracy with a bicameral legislature. Citizens elect representatives through a mixed-member proportional system.
   - The executive branch is led by a president, elected directly for a six-year term.

2